In [1]:
# =============================================================================
# 02b_seed_stability_robustness.ipynb
# =============================================================================
# Purpose:
#   Referee robustness check on whether the constrained regime's advantage is an
#   artefact of a particular random draw. We regenerate counterfactuals at the
#   2-CF setting across multiple DiCE random seeds on a matched subsample and
#   report the mean and spread of proximity, sparsity, and actionability for
#   both regimes. (Note: DiCE's post-hoc sparsity optimisation is left disabled
#   throughout; enabling it is computationally prohibitive at this scale, and the
#   proximity advantage is structural because the constrained search varies a
#   strict subset of the features. Sparsity is measured directly in all cases.)
#
# Inputs:  ../data/modeling_dataset_top30.csv, ../data/selected_features.pkl
# Outputs: ../results/tables/t7_seed_stability_robustness.csv
# =============================================================================

import warnings, time
from pathlib import Path
import numpy as np
import pandas as pd
import joblib
from sklearn.model_selection import train_test_split
import xgboost as xgb
import dice_ml

warnings.filterwarnings("ignore")
TAB_DIR = Path("../results/tables"); TAB_DIR.mkdir(parents=True, exist_ok=True)

SEED = 44
N_SUB = 40
STAT_NCF = 2
SEEDS = [44, 1, 2, 3, 4]
features = joblib.load("../data/selected_features.pkl")
ACTIONABLE = [f for f in features if "Sales" in f]
df = pd.read_csv("../data/modeling_dataset_top30.csv")
X, y = df[features], df["closure"]
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
model = xgb.XGBClassifier(learning_rate=0.01, max_depth=5, min_child_weight=3,
                          n_estimators=100, subsample=0.8, random_state=SEED,
                          eval_metric="logloss").fit(X_tr, y_tr)
mad = {f: max((df[f] - df[f].median()).abs().median(), 1.0) for f in features}
ACT_IDX = [features.index(f) for f in ACTIONABLE]

dice_data = dice_ml.Data(dataframe=df[features + ["closure"]].copy(),
                         continuous_features=features, outcome_name="closure")
explainer = dice_ml.Dice(dice_data, dice_ml.Model(model=model, backend="sklearn"), method="random")

proba = model.predict_proba(df[features])[:, 1]
cand = df.index[(df["closure"] == 1) & (proba >= 0.5)].tolist()
rng = np.random.RandomState(SEED)
sample = sorted(rng.choice(cand, size=min(120, len(cand)), replace=False).tolist())[:N_SUB]
print(f"subsample stores: {len(sample)} | seeds: {SEEDS}")

def pack_metrics(o, C):
    dists, sp, a, tot = [], [], 0, 0
    for _, c in C.iterrows():
        diff = np.abs(o.values - c.values)
        dists.append(np.sum([diff[k] / mad[features[k]] for k in range(len(features))]))
        changed = diff > 1e-6
        sp.append(np.mean(~changed)); tot += int(changed.sum()); a += int(changed[ACT_IDX].sum())
    return -np.mean(dists), np.mean(sp), (a / tot if tot else 1.0)

rows = []
t0 = time.time()
for regime in ["unconstrained", "constrained"]:
    ftv = ACTIONABLE if regime == "constrained" else features
    for sd in SEEDS:
        prox, spars, act, n = [], [], [], 0
        for i in sample:
            q = df[features].iloc[i:i + 1]
            try:
                res = explainer.generate_counterfactuals(
                    q, total_CFs=STAT_NCF, desired_class="opposite",
                    features_to_vary=ftv, random_seed=sd, verbose=False,
                    posthoc_sparsity_param=None)
                C = res.cf_examples_list[0].final_cfs_df
                if C is not None and len(C) > 0:
                    p, s, a = pack_metrics(q.iloc[0], C[features].reset_index(drop=True))
                    prox.append(p); spars.append(s); act.append(a); n += 1
            except Exception:
                pass
        rows.append({"regime": regime, "seed": sd, "n": n,
                     "proximity": np.mean(prox), "sparsity": np.mean(spars),
                     "actionability": np.mean(act)})
print(f"  (generation time: {time.time()-t0:.0f}s)")

raw = pd.DataFrame(rows)
summary = (raw.groupby("regime")[["proximity", "sparsity", "actionability"]]
           .agg(["mean", "std"]).round(3))
summary.columns = [f"{m}_{s}" for m, s in summary.columns]
summary = summary.reset_index()
summary.to_csv(TAB_DIR / "t7_seed_stability_robustness.csv", index=False)
print("\nStability across seeds (mean +/- std over {} seeds):".format(len(SEEDS)))
for _, r in summary.iterrows():
    print(f"  {r['regime']:<13} "
          f"proximity {r['proximity_mean']:.2f}+/-{r['proximity_std']:.2f} | "
          f"sparsity {r['sparsity_mean']:.3f}+/-{r['sparsity_std']:.3f} | "
          f"actionability {r['actionability_mean']:.3f}+/-{r['actionability_std']:.3f}")
print("\nSaved t7_seed_stability_robustness.csv")
print("[NB2b DONE]")


subsample stores: 40 | seeds: [44, 1, 2, 3, 4]


  0%|          | 0/1 [00:00<?, ?it/s]

100%|██████████| 1/1 [00:00<00:00,  6.03it/s]

100%|██████████| 1/1 [00:00<00:00,  5.99it/s]

  0%|          | 0/1 [00:00<?, ?it/s]

...

100%|██████████| 1/1 [00:00<00:00,  7.41it/s]

  0%|          | 0/1 [00:00<?, ?it/s]

100%|██████████| 1/1 [00:00<00:00,  7.31it/s]

100%|██████████| 1/1 [00:00<00:00,  7.27it/s]

  (generation time: 66s)

Stability across seeds (mean +/- std over 5 seeds):
  constrained   proximity -25.46+/-4.95 | sparsity 0.948+/-0.006 | actionability 1.000+/-0.000
  unconstrained proximity -48.35+/-51.31 | sparsity 0.943+/-0.004 | actionability 0.672+/-0.112

Saved t7_seed_stability_robustness.csv
[NB2b DONE]
